# 07 — RunContext and Data Flow

**Related guide:** [07_RUN_CONTEXT_AND_DATA_FLOW.md](../docs/guides/07_RUN_CONTEXT_AND_DATA_FLOW.md)

**Related script:** [07_run_context.py](../examples/07_run_context.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Use `RunContext` to read declared dependency outputs and workflow parameters.

In [ ]:
from pyworkflowkit import (
    RunContext,
    TaskHandle,
    TaskId,
    WorkflowParameter,
    WorkflowRuntime,
    task,
    workflow,
)


@task
def fetch() -> int:
    return 7


@task(depends_on=(fetch,))
def transform(context: RunContext) -> int:
    value = int(context.dependency_outputs[TaskId("fetch")])
    multiplier = int(context.workflow_parameters["multiplier"])
    return value * multiplier


@workflow(
    id="notebooks.run-context",
    version="1",
    parameters=(WorkflowParameter(name="multiplier"),),
)
def demo() -> tuple[TaskHandle, ...]:
    return (fetch, transform)

In [ ]:
runtime = WorkflowRuntime()
for handle in demo.task_handles():
    runtime.register(handle.handler_ref, handle.handler)

run = runtime.run(demo.build(), parameters={"multiplier": 3})
print({"status": run.status.value, "events": len(runtime.events(run.run_id))})

## Experiment

Change `multiplier` and observe that workflow structure stays identical while runtime data changes.